In [1]:
import hydra
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import accuracy_score
import kagglehub

/Users/michellezhou/Documents/workspace/Northwell/spaceship/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def dataset(handle: str) -> None:
    try:
        dataset_dir = kagglehub.dataset_download(handle) 
        print(f'Dataset {handle} downloaded to {dataset_dir}')
    except ValueError:
        dataset_dir = kagglehub.competition_download(handle)
        print(f'Dataset {handle} downloaded to {dataset_dir}')
    except Exception as e:
        print(f'Error downloading dataset: {e}')
    return dataset_dir

def load_train_df(comp):
    return pd.read_csv(comp + '/train.csv')

def load_test_df(comp):
    return pd.read_csv(comp + '/test.csv')

In [3]:
train_df=load_train_df(dataset('spaceship-titanic'))
test_df=load_test_df(dataset('spaceship-titanic'))

Dataset spaceship-titanic downloaded to /Users/michellezhou/.cache/kagglehub/competitions/spaceship-titanic
Dataset spaceship-titanic downloaded to /Users/michellezhou/.cache/kagglehub/competitions/spaceship-titanic


In [4]:
def add_features (train_df, test_df):
    train_df['CabinDeck'] = train_df['Cabin'].str.split('/').str[0]
    train_df['CabinSide'] = train_df['Cabin'].str.split('/').str[2]
    train_df['CabinNumber'] = pd.to_numeric(train_df['Cabin'].str.split('/').str[1])
    train_df['CabinMissing'] = train_df['Cabin'].isna().astype(int)
    train_df['PassengerGroup'] = train_df.PassengerId.str.split('_').str[0].astype(int) 
    train_df['GroupNumber'] = train_df.PassengerId.str.split('_').str[1].astype(int)
    train_df['GroupSize'] = (train_df['PassengerGroup'].map(train_df['PassengerGroup'].value_counts()))
    train_df['SoloTravel'] = (train_df['GroupSize']==1).astype(int)
    train_df['LastName'] = train_df.Name.str.split(' ').str[-1]
    train_df['FamilySize'] = train_df['LastName'].map(train_df['LastName'].value_counts())
    train_df['AgeGroup'] = pd.cut(train_df['Age'], bins=[-1, 5, 13, 18, 60, 100], labels=['Baby', 'Child', 'Teen', 'Adult', 'Elderly'])

    test_df['CabinDeck'] = test_df['Cabin'].str.split('/').str[0]
    test_df['CabinSide'] = test_df['Cabin'].str.split('/').str[2]    
    test_df['CabinNumber'] = pd.to_numeric(test_df['Cabin'].str.split('/').str[1])
    test_df['CabinMissing'] = test_df['Cabin'].isna().astype(int)
    test_df['PassengerGroup'] = test_df.PassengerId.str.split('_').str[0].astype(int)
    test_df['GroupNumber'] = test_df.PassengerId.str.split('_').str[1].astype(int)
    test_df['GroupSize'] = (test_df['PassengerGroup'].map(test_df['PassengerGroup'].value_counts()))
    test_df['SoloTravel'] = (test_df['GroupSize']==1).astype(int)
    test_df['LastName'] = test_df.Name.str.split(' ').str[-1]
    test_df['FamilySize'] = test_df['LastName'].map(test_df['LastName'].value_counts())
    test_df['AgeGroup'] = pd.cut(test_df['Age'], bins=[-1, 5, 13, 18, 60, 100], labels=['Baby', 'Child', 'Teen', 'Adult', 'Elderly'])


    return train_df, test_df

In [5]:
train_df, test_df = add_features(train_df, test_df)

In [6]:
def predict_missing_values(data):
    #No VIP in Earth; No VIP < 18
    data.loc[data['VIP'].isna() & (data['HomePlanet'] == 'Earth'), 'VIP'] = False
    data.loc[data['VIP'].isna() & (data['Age'] < 18), 'VIP'] = False

    #Passengers < 13 and Passengers Sleeping have no spending
    spending_cols = ['Spa', 'ShoppingMall', 'VRDeck', 'FoodCourt', 'RoomService']
    for col in spending_cols:
        data.loc[data[col].isna() & ((data['Age'] < 13 ) | (data['CryoSleep'] == True)), col] = 0

    #Assume groups are families, and have the same last name, and have same destination and home planet, and same cabin deck+side
    group_cols = ['LastName', 'HomePlanet', 'Destination', 'CabinDeck', 'CabinSide']
    for col in group_cols:
        data[col] = data.groupby('PassengerGroup')[col].transform( lambda x: x.ffill().bfill())

    #Anyone who spends is not asleep
    data.loc[data['CryoSleep'].isna() & ((data['Spa']>0) | (data['RoomService']>0) | (data['FoodCourt']>0) | (data['VRDeck']>0) | (data['ShoppingMall']>0)), 'CryoSleep'] = False
       
    return data

In [7]:
train_df = predict_missing_values(train_df)
test_df = predict_missing_values(test_df)

In [8]:
def add_spending (train_df, test_df):
    train_df['TotalSpending'] = (train_df['RoomService'] + train_df['FoodCourt'] + train_df['ShoppingMall'] + train_df['Spa'] + train_df['VRDeck'])
    train_df["SocioEconStatus"] = train_df["TotalSpending"].apply(lambda x: "Low" if x < 5000 else ("Middle" if x < 20000 else "Upper"))
    train_df['LuxurySpend'] = train_df['Spa'] + train_df['VRDeck'] + train_df['RoomService']
    train_df['EssentialSpend'] = train_df['FoodCourt'] + train_df['ShoppingMall']
    train_df['NoSpend'] = (train_df['TotalSpending']==0).astype(int)
    train_df['CryoNoSpend'] = ((train_df['CryoSleep'] == True) & (train_df['NoSpend'] == 1)).astype(int)

    test_df['TotalSpending'] = (test_df['RoomService'] + test_df['FoodCourt'] + test_df['ShoppingMall'] + test_df['Spa'] + test_df['VRDeck'])
    test_df["SocioEconStatus"] = test_df["TotalSpending"].apply(lambda x: "Low" if x < 5000 else ("Middle" if x < 20000 else "Upper"))
    test_df['LuxurySpend'] = test_df['Spa'] + test_df['VRDeck'] + test_df['RoomService']
    test_df['EssentialSpend'] = test_df['FoodCourt'] + test_df['ShoppingMall']
    test_df['NoSpend'] = (test_df['TotalSpending']==0).astype(int)
    test_df['CryoNoSpend'] = ((test_df['CryoSleep'] == True) & (test_df['NoSpend'] == 1)).astype(int)

    return train_df, test_df

In [9]:
train_df, test_df = add_spending(train_df, test_df)

In [10]:
features = ['Age', 'CryoSleep', 'CabinDeck', 'CabinSide', 'CabinNumber', 'GroupSize', 'FamilySize', 'SoloTravel', 'TotalSpending', 'NoSpend', 'LuxurySpend', 'EssentialSpend', 'Destination', 'HomePlanet', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'VIP']
cont_cols=['Age', 'CabinNumber', 'GroupSize', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpending',  'LuxurySpend', 'EssentialSpend', 'FamilySize']
cat_cols=['Destination', 'HomePlanet', 'CryoSleep', 'VIP', 'CabinDeck', 'CabinSide', 'NoSpend', 'SoloTravel']

In [11]:
X = train_df[features]
y = train_df['Transported'].astype(int).to_numpy().flatten()
X_test_raw = test_df[features]

In [12]:
xgb_test_probs = []
cat_test_probs = []
light_test_probs = []
tab_test_probs = []
valid_scores = []
xgb_valid = []
light_valid = []
cat_valid = []
tab_valid = []

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, Normalizer
from sklearn.impute import SimpleImputer

In [14]:
def get_preprocessor():
    cont_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        # ('winsor', Winsorizer(capping_method='iqr', tail='both', fold=1.5)),
        ('normalizer', StandardScaler())
        # ('normalizer', Normalizer())
    ])
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OrdinalEncoder())
    ])
    preprocessor = ColumnTransformer([
        ('cont', cont_pipeline, cont_cols),
        ('cat', cat_pipeline, cat_cols)
    ])
    return preprocessor

In [15]:
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

import torch


def build_tabnet_model():
    model = TabNetClassifier(
                        n_d=16,
                        n_a=16,
                        n_steps=4,
                        gamma=1.5,
                        lambda_sparse=1e-4,
                        optimizer_fn=torch.optim.Adam,
                        optimizer_params=dict(lr=0.02),
                        scheduler_params={"step_size":10, "gamma":0.9},
                        scheduler_fn=torch.optim.lr_scheduler.StepLR,
                        mask_type='entmax',
                        device_name='cpu',
                        verbose= 1,
    )
    return model

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
kf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


: 

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=train_df['PassengerGroup'])):
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y[train_idx]
        y_valid = y[valid_idx]

        #Tabnet
        tab_X_train_raw = X_train.copy()
        tab_X_valid_raw = X_valid.copy()
        tab_X_test_raw = X_test_raw.copy()
        #Normalize
        tab_preprocessor = get_preprocessor()
        tab_preprocessor.fit(tab_X_train_raw)
        tab_X_train = tab_preprocessor.transform(tab_X_train_raw)
        tab_X_valid = tab_preprocessor.transform(tab_X_valid_raw)
        tab_X_test = tab_preprocessor.transform(tab_X_test_raw)
        y_train_tab = y_train.astype(np.int64)
        y_valid_tab = y_valid.astype(np.int64)
        #Model
        tab_model = build_tabnet_model()
        tab_model.fit(tab_X_train, y_train_tab,
                  eval_set = [(tab_X_valid, y_valid_tab)],
                  eval_name = ['valid'],
                  eval_metric = ['accuracy'],
                  max_epochs=100,
                  patience=15,
                  batch_size=512,
                  virtual_batch_size=128,
                  num_workers=0,
                  drop_last=True)
        tab_valid_probs = tab_model.predict_proba(tab_X_valid)[:, 1]
        tab_test_probs.append(tab_model.predict_proba(tab_X_test)[:, 1])


/Users/michellezhou/Documents/workspace/Northwell/spaceship/.venv/lib/python3.11/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
